# Case Study 2 — full pipeline (run top to bottom)

This one notebook runs everything on the university server: build the corpus, embed and index, generate, judge, score, and show the four-bucket result.

**Two switches, set in section 2:**
- **Generator** — the free dev model (Gemini) to shake out bugs now, or the frozen `claude-sonnet-4-6` for the real graded run.
- **Judge** — `stub` (offline, instant) for dev, or the frozen 70B open model via vLLM on this GPU for the real run.

Free-model runs are for **debugging the pipeline, not results**. The numbers that go in the manuscript use the frozen generator + the 70B judge.

Run the cells in order. Sections 3 and 4 build the retrieval store (once). Section 6 generates, section 7 judges and scores, section 8 shows the result.

## 0. Environment probe
Tells us what this server can do. Run it first.

In [ ]:
import sys, os, subprocess, platform, urllib.request
print("python:", sys.version.split()[0], "|", platform.platform())
print("cwd:", os.getcwd())
if not os.path.exists("src/run_generation.py"):
    print("!! Run this notebook from the repo ROOT (the folder with src/, config/, test_set.jsonl).")

def check_internet(url="https://pypi.org", timeout=5):
    try:
        urllib.request.urlopen(url, timeout=timeout); return True
    except Exception as e:
        print("  internet check failed:", e); return False

HAS_INTERNET = check_internet()
print("internet:", HAS_INTERNET)

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        p = torch.cuda.get_device_properties(0)
        print("GPU:", torch.cuda.get_device_name(0),
              f"| VRAM {p.total_memory/1e9:.0f} GB | count {torch.cuda.device_count()}")
    else:
        print("GPU: torch present but no CUDA device visible")
except Exception as e:
    print("GPU: torch not importable yet (install deps in section 1) ->", e)
print(f"\nSUMMARY  internet={HAS_INTERNET}  gpu={HAS_GPU}")

## 1. Install dependencies (run once, needs internet)
vLLM for the 70B judge is heavy and installed later, only when you switch the judge on.

In [ ]:
if HAS_INTERNET:
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
    subprocess.run([sys.executable,"-m","pip","install","-q","openai"], check=False)
    print("core deps installed")
else:
    print("No internet here: install where there is internet, or pre-stage wheels.")

## 2. Config — the only knobs

For the **free dev run** (default): Gemini generator + stub judge. Paste your free Google AI Studio key below.

For the **real graded run**: set `GEN_PROVIDER="anthropic"`, `GEN_MODEL="claude-sonnet-4-6"`, paste `ANTHROPIC_API_KEY`, set `JUDGE="vllm"`, and `RUN_FULL=True`.

In [ ]:
# ---------- GENERATOR ----------
GEN_PROVIDER = "openai_compatible"      # "anthropic" for the frozen graded run
GEN_MODEL    = "gemini-2.0-flash"       # "claude-sonnet-4-6" for the frozen run
GEN_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEN_KEY_ENV  = "GEMINI_API_KEY"
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")   # <-- paste free key
# os.environ["ANTHROPIC_API_KEY"] = ""   # <-- paste for the frozen run

# ---------- JUDGE ----------
JUDGE          = "stub"                  # "vllm" for the real 70B judge on this GPU
JUDGE_MODEL    = "Qwen/Qwen2.5-72B-Instruct"   # or meta-llama/Llama-3.3-70B-Instruct
JUDGE_BASE_URL = "http://localhost:8000/v1"

# ---------- RUN SIZE ----------
RUN_FULL = False    # False = 8-row sanity per config; True = full 215 x 3

def sh(cmd):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    r = subprocess.run(cmd, shell=not isinstance(cmd, list), capture_output=True, text=True)
    if r.stdout: print(r.stdout[-6000:])
    if r.returncode != 0 and r.stderr: print("STDERR:\n", r.stderr[-4000:])
    return r.returncode

print("generator:", GEN_PROVIDER, GEN_MODEL, "| judge:", JUDGE, "| full run:", RUN_FULL)

## 3. Corpus (400 chunks from the frozen guidelines)
Uses the committed corpus if present; only rebuilds if missing (rebuild needs poppler/pdftotext).

In [ ]:
CHUNKS = "results/corpus_chunks.jsonl"
PDF = "data/guidelines/Draft_Guidelines_on_the_classification_of_high_risk_AI_Annex_III.pdf"
if os.path.exists(CHUNKS):
    print("corpus present (committed):", sum(1 for _ in open(CHUNKS)), "chunks — skip rebuild")
else:
    sh([sys.executable, "src/build_corpus.py", PDF, CHUNKS])
    print("chunks:", sum(1 for _ in open(CHUNKS)))

## 4. Embed + index (downloads bge model, needs internet, builds Chroma)
The vector store is not in git, so build it here once.

In [ ]:
sh([sys.executable, "src/embed_and_index.py", "config/pipeline.yaml"])

## 5. Retrieval quality check (optional, no API needed)
Should show Hit@5 around 0.83.

In [ ]:
sh([sys.executable, "src/retrieval_eval.py", "config/pipeline.yaml"])

## 6. Generation
Writes one file per config to results/runs/. Sanity (8 rows) unless RUN_FULL=True.

In [ ]:
gen_args = [sys.executable, "src/run_generation.py",
            "--provider", GEN_PROVIDER, "--model", GEN_MODEL,
            "--base-url", GEN_BASE_URL, "--api-key-env", GEN_KEY_ENV]
if not RUN_FULL:
    gen_args += ["--limit", "8"]
sh(gen_args)

import glob, json
for f in sorted(glob.glob("results/runs/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), "->", len(rows), "rows | first: pred=%s gold=%s" %
          (rows[0]["pred_label"], rows[0]["gold_label"]))

## 7. Judge + scoring
`stub` = offline and instant (dev). `vllm` = the real 70B judge on this GPU: the next cell starts a vLLM server (first run downloads the 70B, can take a while), scores against it, then stops it.

In [ ]:
vllm_proc = None
if JUDGE == "vllm":
    if not HAS_GPU:
        print("JUDGE=vllm but no GPU detected. Switch JUDGE to 'stub' or run on the GPU node.")
    else:
        subprocess.run([sys.executable,"-m","pip","install","-q","vllm"], check=False)
        import time, urllib.request
        vllm_proc = subprocess.Popen([sys.executable,"-m","vllm.entrypoints.openai.api_server",
            "--model", JUDGE_MODEL, "--port", "8000", "--dtype", "auto"])
        print("starting vLLM (first load downloads the 70B weights)...")
        for _ in range(180):
            try:
                urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3)
                print("vLLM server is up"); break
            except Exception:
                time.sleep(10)
else:
    print("JUDGE=stub — no model needed; scoring runs offline and instant.")

In [ ]:
score_args = [sys.executable, "src/run_scoring.py", "--judge", JUDGE]
if JUDGE == "vllm":
    score_args += ["--judge-model", JUDGE_MODEL, "--judge-base-url", JUDGE_BASE_URL]
sh(score_args)

if vllm_proc is not None:
    vllm_proc.terminate(); print("vLLM server stopped")

## 8. Results — the four-bucket matrix
Correctness, faithfulness, then the right/wrong x faithful/unfaithful buckets. The off-diagonal rows are in results/scoring/buckets/*_offdiagonal.jsonl.

In [ ]:
for f in ["results/scoring/correctness_summary.md",
          "results/scoring/faithfulness_summary.md",
          "results/scoring/buckets_summary.md"]:
    print("="*72); print(f); print("="*72)
    print(open(f).read() if os.path.exists(f) else "(not produced yet)")
    print()